In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 

In [0]:
%run /Workspace/Users/malni1880@gmail.com/FMCG_project/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","produts","Data Source")

catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

In [0]:
base_path=f'/Volumes/fmcg/source_data/raw/{data_source}.csv'
print(base_path)

### BRONZE PROCESSING

In [0]:
df= (
  spark.read.format("csv")
  .option("header",True)
  .option("inferSchema",True)
  .load(base_path)
  .withColumn("read_timestamp",F.current_timestamp())
  .select("*","_metadata.file_name","_metadata.file_size"))


In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df.write.format("delta").option("Delta.enableChangeDataFeed","true").mode("overwrite").saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### SILVER PROCESSING

In [0]:
df_bronze= spark.sql("select * from fmcg.bronze.products;")
df_bronze.show(10)

In [0]:
print("Rows before duplicates dropped: ",df_bronze.count())
df_silver= df_bronze.dropDuplicates()
print("Rows after duplicates dropped: ",df_silver.count())

In [0]:
df_silver=df_silver.withColumn("category",F.when(F.col("category").isNull(),None).otherwise(F.initcap("category")))

In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver=(df_silver.withColumn("product_name",F.regexp_replace("product_name","(?i)protien","Protein"))
           .withColumn("category",F.regexp_replace("category","(?i)protien","Protein")))
           


In [0]:
df_silver=(
    df_silver.withColumn("division",F.when(F.col("category")=="Energy Bars", "Nutrition Bars")
                        .when(F.col("category")=="Protein Bars", "Nutrition Bars")
                        .when(F.col("category")=="Granola & Cereals", "Breakfast Foods")
                        .when(F.col("category")=="Recovery Dairy", "Dairy & Recovery")
                        .when(F.col("category")=="Healthy Snacks", "Healthy Snacks")
                        .when(F.col("category")=="Electrolyte Mix", "Hydration & Electrolytes")
                        .otherwise("Other")
                        )
)

df_silver=df_silver.withColumn("variant",F.regexp_extract(F.col("product_name"),r"\((.*?)\)",1))

In [0]:
display(df_silver.limit(10))

In [0]:
df_silver=df_silver.withColumn("product_code",F.sha2(F.col("product_name").cast("string"), 256))

df_silver=df_silver.withColumn(
    "product_id",F.when(F.col("product_id").cast("string").rlike("^[0-9]+$"),
    F.col("product_id").cast("string"))
    .otherwise(F.lit("999999").cast("string"))
)

df_silver=df_silver.withColumnRenamed("product_name","product")

In [0]:
display(df_silver)

In [0]:
df_silver=df_silver.select("product_code","division","category","product","variant","product_id","read_timestamp","file_name","file_size")
display(df_silver)

In [0]:
df_silver.write.format("delta").mode("overwrite")\
    .option("Delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### GOLD PROCESSING


In [0]:
df_silver=spark.sql("select * from fmcg.silver.products")

In [0]:
df_gold=df_silver.select("product_code","product_id","division","category","product","variant")
df_gold.show(5)

In [0]:
df_gold.write.format("delta").mode("overwrite")\
    .option("mergeSchema", "true")\
    .option("Delta.enableChangeDataFeed", "true")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

### MERGING DATA WITH PARENT

In [0]:
delta_table=DeltaTable.forName(spark,"fmcg.gold.dim_products")
df_child_products=spark.sql("select product_code,division,category,product,variant from fmcg.gold.sb_dim_products")

In [0]:
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.product_code==source.product_code"
).whenMatchedUpdate(
    set={"division": "source.division",
         "category": "source.category",
         "product": "source.product",
         "variant": "source.variant"}
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
         "category": "source.category",
         "product": "source.product",
         "variant": "source.variant"
    }
).execute()